<font color='darkred'> Unless otherwise noted, **this notebook will not be reviewed or autograded.**</font> You are welcome to use it for scratchwork, but **only the files listed in the exercises will be checked.**

---

# Background

Before we move into the exercises, we'll cover some background information to get you started.

In [1]:
import numpy as np
import pandas as pd
import requests
import re

## Markov Chain Simple Example

Markov chains are a way of representing how systems change over time. The main concept behind Markov chains are that they are memoryless, meaning that the next state of a process only depends on the previous state.

![image](https://upload.wikimedia.org/wikipedia/commons/7/7a/Markov_Chain_weather_model_matrix_as_a_graph.png)

The way to read the Markov chain above from [Wikipedia](https://commons.wikimedia.org/w/index.php?curid=25300524) is:
* If I am currently in the sunny state, there is a 10% chance I will go to the rainy state and a 90% chance I will remain in the sunny state
* If I am currently in the rainy state, there is an 50% chance I will go to the sunny state and a 50% chance I will remain in the rainy state

## Transition Matrices

This is what our **transition matrix** will look like for the Markov chain diagram above. Take a minute to interpret the rows and columns of this matrix.

In [2]:
P = np.asarray([.9, .1, .5, .5]).reshape(2,2)
states = ['sunny', 'rainy']

pd.DataFrame(P, index=states, columns=states)

,sunny,rainy
sunny,0.9,0.1
rainy,0.5,0.5


## Predict Tomorrow's Weather

Let's say it's sunny today, we can represent that as:

`today = [1, 0]`

**Predict tomorrow's weather using what you know about today and the transition matrix.**

In [3]:
today = [1, 0]

tomorrow = np.dot(today, P)
tomorrow

array([0.9, 0.1])

In this example, there is a 90% chance it will remain sunny tomorrow, and a 10% chance it'll be rainy.

**Predict the day after tomorrow's weather.**

In [4]:
# Method 1: Multiply tomorrow's weather by the transition matrix
day_after = np.dot(tomorrow, P)
day_after

array([0.86, 0.14])

In [5]:
# Method 2: Multiply today's weather by the transition matrix^2
day_after = np.dot(today, np.linalg.matrix_power(P, 2))
day_after

array([0.86, 0.14])

## Text Generation

Markov chains can also be used for very basic text generation. **Think about every word in a corpus as a state.** We can make a simple assumption that the next word is only dependent on the previous word - which is the basic assumption of a Markov chain. In this exercise, you'll create a text generator which uses only this concept.

## Read in some text to imitate

We are going to generate some text in the style of inspirational quotes, so let's first read in the data.

In [6]:
url = 'https://raw.githubusercontent.com/leontoddjohnson/datasets/main/text/inspiration_quotes.txt'

content = requests.get(url, timeout=30)
content.raise_for_status()
quotes_raw = content.text

print(quotes_raw[:1000])

“Healing comes from taking responsibility: to realize that it is you - and no one else - that creates your thoughts, your feelings, and your actions.” —Peter Shepherd

“Life is a journey and if you fall in love with the journey you will be in love forever.” —Peter Hagerty

“When you return to your old hometown, you find it wasn’t the town you missed, but your childhood.” —Earl Wilson

“As we grow old, the beauty steals inward.” —Ralph Waldo Emerson

“Life begins as a quest of the child for the man, and ends as a journey by the man to rediscover the child.” —Sam Ewing

Happiness
“Ultimately your greatest teacher is to live with an open heart.” —Emmanuel (Pat Rodegast)

“Doing what you like is freedom. Liking what you do is happiness.” —Frank Tyger

“We forge the chains we wear in life.” —Charles Dickens

happiness quote
“If you look to others for fulfillment, you will never be fulfilled. If your happiness depends on money, you will never be happy with yourself. Be content with what you 

## Clean up the text data

There are many ways to clean up data before building a text generator. In this case, we'll try to at least just extract the quotes themselves.

*After you complete the exercises, feel free to adjust this section of the process ...*

In [7]:
quotes = quotes_raw.replace('\n', ' ')
quotes = re.split("[“”]", quotes)   # split on the unique “ characters
quotes[:3]

['',
 'Healing comes from taking responsibility: to realize that it is you - and no one else - that creates your thoughts, your feelings, and your actions.',
 ' —Peter Shepherd  ']

In [8]:
# skip the first one, and capture every other element
quotes = quotes[1::2]
quotes[:3]

['Healing comes from taking responsibility: to realize that it is you - and no one else - that creates your thoughts, your feelings, and your actions.',
 'Life is a journey and if you fall in love with the journey you will be in love forever.',
 'When you return to your old hometown, you find it wasn’t the town you missed, but your childhood.']

In [9]:
# create one long corpus of text
corpus = ' '.join(quotes)

# remove long whitespaces (see regex101.com)
corpus = re.sub(r"\s+", " ", corpus)

# remove leading/trailing whitespaces
corpus = corpus.strip()

corpus[:200]

'Healing comes from taking responsibility: to realize that it is you - and no one else - that creates your thoughts, your feelings, and your actions. Life is a journey and if you fall in love with the '

In general, this version of `corpus` should work just fine!

# Exercises

In these exercises, you'll build a `MarkovText` object that takes in a `corpus` of text, and has the capability to generate text using the Markov Property.

In [10]:
from apputil import *

## Exercise 1: Build a Transition Dictionary

Update the `get_term_dict` method to build a term dictionary of Markov states with the following traits:

* The keys should be (unique) tokens in the corpus
* For each key, the value should be a `list` of all the tokens that follow that key
    - E.g., if my total corpus is "Astrid is very kind, is she not?", then my dictionary should include `{... "is": ["very", "she"] ...}`.
    - Decide whether or not to include duplicates (i.e., *every* iteration) in these lists. Then, explain why or why not.

*Hint: You'll likely want to use [`defaultdict(list)`](https://realpython.com/python-defaultdict/#understanding-the-python-defaultdict-type) here.*

Apply the function to the quotes above. Your final output should look something like this:
    
```python
{
    'Healing': ['comes'],
    'comes': ['from', 'the', 'the', ...],
    'from': ['taking', 'aesthetic', 'a'],
    ...
}
```

In [11]:
text_gen = MarkovText(corpus)
term_dict = text_gen.get_term_dict()

# Preview the first ten states; the full dictionary is stored on text_gen.
dict(list(term_dict.items())[:10])

{'Healing': ['comes'],
 'comes': ['from', 'from'],
 'from': ['taking',
  'aesthetic',
  'a',
  'having',
  'not',
  'the',
  'the',
  'it.',
  'our',
  'experience',
  'experience.',
  'avoiding',
  'our'],
 'taking': ['responsibility:', 'it.'],
 'responsibility:': ['to'],
 'to': ['realize',
  'your',
  'rediscover',
  'live',
  'others',
  'you.',
  'be',
  'be',
  'pay',
  'give',
  'share',
  'have',
  'draw',
  'think',
  'you;',
  'go',
  'the',
  'obtain',
  'mourn',
  'live',
  'a',
  'make',
  'forgive.',
  'take',
  'do.',
  'be',
  'be',
  'change',
  'control',
  'an',
  'take',
  'accept',
  'change',
  'know',
  'win,',
  'be',
  'succeed,',
  'live',
  'the',
  'order,',
  'clarity.',
  'get',
  'be',
  'shine',
  'manifest',
  'do',
  'work',
  'work',
  'think',
  'play',
  'read',
  'be',
  'happiness',
  'love',
  'share',
  'be',
  'laugh',
  'dream',
  'a',
  'make',
  'others;',
  'prove',
  'learn',
  'focus',
  'anything,',
  'accept',
  'change',
  'know',
  'tr

### Transition dictionary decision

I keep duplicate successors because their counts preserve the observed transition frequencies. For `"a b a b a c"`, the entry for `a` is `["b", "b", "c"]`. Sampling from this list gives `b` probability `2/3` and `c` probability `1/3`. Removing duplicates would change both probabilities to `1/2`.

I use whitespace tokenization and preserve capitalization and punctuation. Every unique token is a key, including a final token with no successor; its list is empty. Rebuilding the dictionary starts from scratch.

## Exercise 2: Create a text generator

Update the `generate()` method to generate sentences using the Markov property.

- This should use the `.term_dict`, and it should take in the number of terms you want generated, `term_count`.
- Your function should also be able to accept an *optional* user-defined `seed_term` that starts the generator. By default, you can have the function select a random first token.
  - If the user-defined seed term is not in the corpus, raise a `ValueError`.
- Each "next word" should be chosen *at random* given the "current" word and the term dictionary.
  - Consider [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html) for this.

*Hint: think about what happens if the last term in the corpus occurs only **once**.*

In [12]:
text_gen.generate()

'Frankl happiness is being who they can have to get report cards in many parts.'

### Generation decisions and checks

`term_count` includes the starting word and defaults to 15. The optional `seed_term` must match a corpus token exactly; an unknown seed raises `ValueError`. Each observed successor is sampled with `random.choice`, retaining the frequency weighting above.

When the current word has no observed successor, I restart from a random unique corpus token so generation still reaches the requested length. This fallback is an explicit modeling choice, not an observed word pair. Zero terms returns an empty string; a positive request from an empty corpus raises `ValueError`.

In [13]:
seed_term = next(iter(text_gen.term_dict))
generated = text_gen.generate(seed_term=seed_term, term_count=20)
print("Seed:", seed_term)
print(generated)
assert generated.split()[0] == seed_term
assert len(generated.split()) == 20

Seed: Healing
Healing comes from the future. The right way things I cannot judge what you did not learning from not balanced


In [14]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "test_apputil.py", "-v"],
    capture_output=True, text=True,
)
print(result.stdout + result.stderr)
result.check_returncode()

test_empty_corpus_and_term_counts (__main__.TestMarkovText.test_empty_corpus_and_term_counts) ... ok
test_generation_uses_frequency_list (__main__.TestMarkovText.test_generation_uses_frequency_list) ... ok
test_invalid_seed (__main__.TestMarkovText.test_invalid_seed) ... ok
test_seed_length_and_observed_transitions (__main__.TestMarkovText.test_seed_length_and_observed_transitions) ... ok
test_single_word_and_whitespace (__main__.TestMarkovText.test_single_word_and_whitespace) ... ok
test_terminal_word_restarts (__main__.TestMarkovText.test_terminal_word_restarts) ... ok
test_transition_dictionary (__main__.TestMarkovText.test_transition_dictionary) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.001s

OK



*Note: this exercise illustrates both a Markov Chain (with constant transition probabilities) **and** a [Monte Carlo Simulation](https://en.wikipedia.org/wiki/Monte_Carlo_method) (iterative sampling from a constantly defined probability distribution of words)!*

---

# Bonus Exercise

*<font color='darkorange'>The following exercise is completely optional</font>, and will not be reviewed unless explicitly requested. That said, these are interesting to work on if you have the time!*

## Part 1

Try to incorporate recursion into your text generator class.

## Part 2

Adjust your text generator such that we can increase the "state window size" from one word to `k` words. That is, the term dictionary in this version with `k = 2` would look something like this (*note the tuple keys*):

```python
{
    ("Healing", "comes"): ["from", ...],
    ...,
    ("it", "is"): ["you", "very"],
    ...
}
```

Generate some text for `k` values of 1, 2, and 3, and compare the differences in text coherence. Try larger corpuses ...